# 📘 Notebook 4: Class Methods & Static Methods

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the three types of methods: **instance**, **class**, and **static**
2. Use the **`@classmethod`** decorator for alternative constructors
3. Use the **`@staticmethod`** decorator for utility functions
4. Know **when to use** each type of method
5. Build a **factory method** pattern with `@classmethod`

---

## 1. Three Types of Methods

| Method Type | Decorator | First Parameter | Can Access | When to Use |
|-------------|-----------|----------------|------------|-------------|
| **Instance Method** | None | `self` (instance) | Instance & class attributes | Actions on a specific object |
| **Class Method** | `@classmethod` | `cls` (class) | Class attributes only | Alternative constructors, factory methods |
| **Static Method** | `@staticmethod` | None (regular) | Nothing (self-contained) | Utility functions related to the class |

```python
class Item:
    @classmethod                    # ← Decorator
    def from_csv(cls):              # ← 'cls' is the class itself
        ...
    
    @staticmethod                   # ← Decorator
    def is_integer(num):            # ← No self or cls
        ...
    
    def calculate_total(self):      # ← Regular instance method
        ...
```

---

## 2. Instance Methods (Review)

These are the methods you've been using. They receive `self` (the instance) as the first argument:

In [1]:
class Item:
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    # Instance method — operates on a SPECIFIC instance via 'self'
    def calculate_total_price(self):
        return self.price * self.quantity
    
    # Another instance method
    def apply_discount(self, discount_percent: float):
        self.price = self.price * (1 - discount_percent / 100)

item = Item("Phone", 100, 5)
print(f"Total: ${item.calculate_total_price()}")
item.apply_discount(20)
print(f"After 20% off: ${item.price}")

Total: $500
After 20% off: $80.0


---

## 3. Class Methods with `@classmethod`

A **class method** receives the **class itself** (not an instance) as its first argument, conventionally named `cls`.

### When to use:
- **Alternative constructors** — different ways to create objects
- **Factory methods** — creating instances from different data sources
- Modifying **class-level** state

### The `cls` Parameter

```
Item.from_csv("data.csv")
  │
  └── Python passes Item as 'cls' automatically
      → from_csv(cls=Item, filename="data.csv")
```

In [2]:
import csv
import os

class Item:
    pay_rate = 0.8
    all = []
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        assert price >= 0, f"Price {price} is not >= 0!"
        assert quantity >= 0, f"Quantity {quantity} is not >= 0!"
        self.name = name
        self.price = price
        self.quantity = quantity
        Item.all.append(self)
    
    def calculate_total_price(self):
        return self.price * self.quantity
    
    @classmethod
    def instantiate_from_csv(cls, filepath: str):
        """Alternative constructor: create items from a CSV file.
        
        cls refers to the class (Item), not an instance.
        We use cls() instead of Item() so subclasses work correctly.
        """
        with open(filepath, 'r') as f:
            reader = csv.DictReader(f)
            items = list(reader)
        
        for item in items:
            cls(  # ← Using cls(), not Item()!
                name=item.get('name'),
                price=float(item.get('price')),
                quantity=int(item.get('quantity')),
            )
    
    def __repr__(self):
        return f"Item('{self.name}', {self.price}, {self.quantity})"

# First, let's create a CSV file to work with
csv_content = """name,price,quantity
Phone,100,5
Laptop,1000,3
Cable,10,50
Mouse,50,10
Keyboard,75,8"""

with open('items.csv', 'w') as f:
    f.write(csv_content)

# Use the class method to create items from CSV
Item.instantiate_from_csv('items.csv')

print("Items loaded from CSV:")
for item in Item.all:
    print(f"  {item} → Total: ${item.calculate_total_price()}")

# Clean up
os.remove('items.csv')

Items loaded from CSV:
  Item('Phone', 100.0, 5) → Total: $500.0
  Item('Laptop', 1000.0, 3) → Total: $3000.0
  Item('Cable', 10.0, 50) → Total: $500.0
  Item('Mouse', 50.0, 10) → Total: $500.0
  Item('Keyboard', 75.0, 8) → Total: $600.0


### More Alternative Constructor Examples

Alternative constructors are one of the most common uses of `@classmethod`:

In [3]:
from datetime import date

class Employee:
    def __init__(self, name: str, age: int, salary: float):
        self.name = name
        self.age = age
        self.salary = salary
    
    @classmethod
    def from_birth_year(cls, name: str, birth_year: int, salary: float):
        """Create an Employee using birth year instead of age."""
        age = date.today().year - birth_year
        return cls(name, age, salary)  # Returns a new Employee instance
    
    @classmethod
    def from_string(cls, employee_string: str):
        """Create an Employee from a dash-separated string.
        Format: 'Name-Age-Salary'
        """
        name, age, salary = employee_string.split('-')
        return cls(name, int(age), float(salary))
    
    def __repr__(self):
        return f"Employee('{self.name}', age={self.age}, salary=${self.salary:,.0f})"

# Standard constructor
emp1 = Employee("Alice", 30, 85000)

# Alternative: from birth year
emp2 = Employee.from_birth_year("Bob", 1990, 92000)

# Alternative: from string
emp3 = Employee.from_string("Charlie-28-78000")

print(emp1)
print(emp2)
print(emp3)

Employee('Alice', age=30, salary=$85,000)
Employee('Bob', age=36, salary=$92,000)
Employee('Charlie', age=28, salary=$78,000)


### Why `cls()` instead of `Item()`?

Using `cls()` makes class methods work correctly with **inheritance**:

In [4]:
class Animal:
    def __init__(self, name: str, sound: str):
        self.name = name
        self.sound = sound
    
    @classmethod
    def create_unknown(cls):
        """Create an animal with unknown attributes."""
        return cls("Unknown", "???")  # cls is whatever class calls this
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', '{self.sound}')"

class Dog(Animal):
    pass

# When called on Dog, cls = Dog (not Animal)
unknown_dog = Dog.create_unknown()
print(f"{unknown_dog} → type: {type(unknown_dog).__name__}")
# Output: Dog('Unknown', '???') → type: Dog  ✅ (not Animal!)

Dog('Unknown', '???') → type: Dog


---

## 4. Static Methods with `@staticmethod`

A **static method** doesn't receive `self` or `cls`. It's just a regular function that lives inside the class for organizational purposes.

### When to use:
- Utility/helper functions **logically related** to the class
- Functions that **don't need** access to instance or class data
- When the function **could be standalone** but belongs conceptually with the class

### Decision Rule:
> **If the method doesn't use `self` or `cls`, it should be a `@staticmethod`.** If it uses `self`, it's an instance method. If it uses `cls`, it's a `@classmethod`.

In [5]:
class Item:
    @staticmethod
    def is_integer(num):
        """Check if a number is an integer (including float like 5.0).
        
        This is a utility function — it doesn't need 'self' or 'cls'.
        It's related to Item (for price validation), so it lives here.
        """
        if isinstance(num, float):
            return num.is_integer()  # 5.0 → True, 5.5 → False
        elif isinstance(num, int):
            return True
        else:
            return False

# Can be called on the class OR an instance
print(f"Item.is_integer(5)   = {Item.is_integer(5)}")
print(f"Item.is_integer(5.0) = {Item.is_integer(5.0)}")
print(f"Item.is_integer(5.5) = {Item.is_integer(5.5)}")
print(f'Item.is_integer("5") = {Item.is_integer("5")}')

# Also works on instances (but doesn't use self)
item = Item()
print(f"\nitem.is_integer(10) = {item.is_integer(10)}")

Item.is_integer(5)   = True
Item.is_integer(5.0) = True
Item.is_integer(5.5) = False
Item.is_integer("5") = False

item.is_integer(10) = True


### More Static Method Examples

In [6]:
class MathHelper:
    """A class that groups related math utility functions."""
    
    @staticmethod
    def add(a, b):
        return a + b
    
    @staticmethod
    def is_even(num):
        return num % 2 == 0
    
    @staticmethod
    def factorial(n):
        if n <= 1:
            return 1
        return n * MathHelper.factorial(n - 1)
    
    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return (celsius * 9/5) + 32

print(f"add(3, 5) = {MathHelper.add(3, 5)}")
print(f"is_even(7) = {MathHelper.is_even(7)}")
print(f"factorial(5) = {MathHelper.factorial(5)}")
print(f"100°C = {MathHelper.celsius_to_fahrenheit(100)}°F")

add(3, 5) = 8
is_even(7) = False
factorial(5) = 120
100°C = 212.0°F


---

## 5. Side-by-Side Comparison

Let's see all three method types in one class:

In [7]:
class Item:
    pay_rate = 0.8
    all = []
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
        Item.all.append(self)
    
    # === INSTANCE METHOD ===
    # Uses 'self' → operates on a specific instance
    def calculate_total_price(self):
        return self.price * self.quantity
    
    def apply_discount(self):
        self.price = self.price * self.pay_rate
    
    # === CLASS METHOD ===
    # Uses 'cls' → operates on the class itself
    @classmethod
    def from_dict(cls, data: dict):
        """Alternative constructor from a dictionary."""
        return cls(
            name=data['name'],
            price=data['price'],
            quantity=data.get('quantity', 0)
        )
    
    @classmethod
    def set_pay_rate(cls, new_rate: float):
        """Modify class-level state."""
        cls.pay_rate = new_rate
    
    # === STATIC METHOD ===
    # No self or cls → standalone utility
    @staticmethod
    def is_integer(num):
        """Utility function — doesn't need self or cls."""
        if isinstance(num, float):
            return num.is_integer()
        return isinstance(num, int)
    
    def __repr__(self):
        return f"Item('{self.name}', {self.price}, {self.quantity})"

# Using instance method
item1 = Item("Phone", 100, 5)
print(f"Instance method: total = ${item1.calculate_total_price()}")

# Using class method (alternative constructor)
item2 = Item.from_dict({"name": "Laptop", "price": 1000, "quantity": 3})
print(f"Class method:    {item2}")

# Using class method (modifying class state)
Item.set_pay_rate(0.9)
print(f"New pay rate:    {Item.pay_rate}")

# Using static method
print(f"Static method:   is_integer(5.0) = {Item.is_integer(5.0)}")

Instance method: total = $500
Class method:    Item('Laptop', 1000, 3)
New pay rate:    0.9
Static method:   is_integer(5.0) = True


---

## 6. Decision Flowchart: Which Method Type?

```
Does the method need access to instance data (self.name, self.price)?
    │
    ├── YES → Use an INSTANCE METHOD (def method(self, ...))
    │
    └── NO → Does the method need access to class data (cls.pay_rate, cls())?
              │
              ├── YES → Use a CLASS METHOD (@classmethod, def method(cls, ...))
              │
              └── NO → Use a STATIC METHOD (@staticmethod, def method(...))
```

### Quick Reference:

| Use Case | Method Type |
|----------|-------------|
| Calculate total for THIS item | Instance method |
| Create an Item from a CSV file | Class method |
| Check if a number is an integer | Static method |
| Apply discount to THIS item | Instance method |
| Change the discount rate for ALL items | Class method |
| Validate an email format | Static method |

---

## 🏋️ Practice Exercises

### Exercise 1: `Date` class with alternative constructors
Create a `Date` class with:
- `__init__(self, year, month, day)`
- `@classmethod from_string(cls, date_string)` — parses "YYYY-MM-DD"
- `@classmethod today(cls)` — creates a Date for today
- `@staticmethod is_valid_date(year, month, day)` — validates date values

In [ ]:
# Exercise 1: Your code here



### Exercise 2: `Pizza` factory
Create a `Pizza` class with:
- `__init__(self, name, size, toppings, price)`
- `@classmethod margherita(cls, size)` — creates a Margherita pizza
- `@classmethod pepperoni(cls, size)` — creates a Pepperoni pizza
- `@staticmethod calculate_area(diameter)` — returns the area

In [ ]:
# Exercise 2: Your code here



---

### ✅ Solutions

In [8]:
# Solution 1: Date class
from datetime import date

class Date:
    def __init__(self, year: int, month: int, day: int):
        if not Date.is_valid_date(year, month, day):
            raise ValueError(f"Invalid date: {year}-{month}-{day}")
        self.year = year
        self.month = month
        self.day = day
    
    @classmethod
    def from_string(cls, date_string: str):
        """Parse a date from 'YYYY-MM-DD' format."""
        year, month, day = map(int, date_string.split('-'))
        return cls(year, month, day)
    
    @classmethod
    def today(cls):
        """Create a Date for today."""
        t = date.today()
        return cls(t.year, t.month, t.day)
    
    @staticmethod
    def is_valid_date(year: int, month: int, day: int) -> bool:
        """Check if the given date values are valid."""
        if month < 1 or month > 12:
            return False
        if day < 1 or day > 31:
            return False
        if year < 1:
            return False
        return True
    
    def __repr__(self):
        return f"Date({self.year}, {self.month}, {self.day})"

d1 = Date(2024, 3, 15)
d2 = Date.from_string("2024-12-25")
d3 = Date.today()

print(f"Manual:  {d1}")
print(f"String:  {d2}")
print(f"Today:   {d3}")
print(f"Valid?   {Date.is_valid_date(2024, 13, 1)}")  # False

Manual:  Date(2024, 3, 15)
String:  Date(2024, 12, 25)
Today:   Date(2026, 6, 2)
Valid?   False


In [9]:
# Solution 2: Pizza factory
import math

class Pizza:
    def __init__(self, name: str, size: int, toppings: list, price: float):
        self.name = name
        self.size = size  # diameter in inches
        self.toppings = toppings
        self.price = price
    
    @classmethod
    def margherita(cls, size: int = 12):
        """Create a Margherita pizza."""
        prices = {10: 8.99, 12: 10.99, 14: 13.99}
        return cls("Margherita", size, ["mozzarella", "tomato", "basil"], prices.get(size, 10.99))
    
    @classmethod
    def pepperoni(cls, size: int = 12):
        """Create a Pepperoni pizza."""
        prices = {10: 9.99, 12: 12.99, 14: 15.99}
        return cls("Pepperoni", size, ["mozzarella", "pepperoni", "tomato sauce"], prices.get(size, 12.99))
    
    @staticmethod
    def calculate_area(diameter: float) -> float:
        """Calculate the area of a pizza given its diameter."""
        radius = diameter / 2
        return math.pi * radius ** 2
    
    def __repr__(self):
        return f"Pizza('{self.name}', {self.size}\", {self.toppings}, ${self.price})"

p1 = Pizza.margherita(14)
p2 = Pizza.pepperoni(12)

print(p1)
print(p2)
print(f"\n14\" pizza area: {Pizza.calculate_area(14):.1f} sq inches")
print(f"12\" pizza area: {Pizza.calculate_area(12):.1f} sq inches")

Pizza('Margherita', 14", ['mozzarella', 'tomato', 'basil'], $13.99)
Pizza('Pepperoni', 12", ['mozzarella', 'pepperoni', 'tomato sauce'], $12.99)

14" pizza area: 153.9 sq inches
12" pizza area: 113.1 sq inches


---

## 📌 Key Takeaways

1. **Instance methods** use `self` → operate on specific objects
2. **Class methods** use `cls` → alternative constructors, modify class state
3. **Static methods** use neither → standalone utility functions grouped with the class
4. Use `cls()` (not `ClassName()`) inside class methods for **inheritance** compatibility
5. If you're unsure, ask: "Does this function need `self` or `cls`?" If not, make it `@staticmethod`

---

**⏭️ Next: [Notebook 05 — Inheritance](./05_Inheritance.ipynb)** — Learn how classes can inherit attributes and methods from parent classes!